In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Imports, Paths, and Global Constants
# ═══════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import pandas as pd, numpy as np, json, time, random, itertools, cv2
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import warnings; warnings.filterwarnings("ignore")

DATA_ROOT  = Path("/kaggle/input/datasets/ashery/chexpert")
TRAIN_CSV  = Path("/kaggle/input/datasets/ashery/chexpert/train.csv")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME        = "densenet121"
IMAGE_SIZE        = 224
NUM_CLASSES       = 14
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED              = 42
HP_SEARCH_SUBSET  = 0.10   # 10% for HP search
HP_SEARCH_EPOCHS  = 5
FINAL_EPOCHS      = 15
HP_PATIENCE       = 3
FINAL_PATIENCE    = 5
MAX_TRIALS        = 36     # Random search (Bergstra & Bengio, 2012)
USE_CLAHE         = True   # CLAHE is fixed — shown to improve CXR contrast

LABEL_NAMES = [
    "No Finding","Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity",
    "Lung Lesion","Edema","Consolidation","Pneumonia","Atelectasis",
    "Pneumothorax","Pleural Effusion","Pleural Other","Fracture","Support Devices"
]
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}  |  CLAHE: {USE_CLAHE}")

Device: cuda  |  Model: densenet121  |  CLAHE: True


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Literature-Justified HP Search Space
# ═══════════════════════════════════════════════════════════════════════
# References:
#  [D1] CheXpert/CheXNeXt baseline  — Adam lr=1e-4, batch=16, constant LR
#  [D2] Idiap NIH-CXR14 grid search — lr best at 5e-5 (scratch) or 1e-4 (fine-tune)
#  [D3] COVID-19 CXR DenseNet study  — Adam lr=1e-3/1e-4, batch=32, strong augment
#  [D4] DR / fundus DenseNet study   — dropout 0.2–0.5, batch=32, 40 epochs Adam
#  [D5] Clinical augmentation paper  — histogram norm + Gaussian noise improves AUROC
HP_SPACE = {
    "learning_rate" : [5e-5, 1e-4, 5e-4],  # [D1] centre: 1e-4; [D2] extends to 5e-5; [D3] up to 5e-4
    "batch_size"    : [16, 32, 48],          # [D1] 16; [D3] 32; current code 48 = GPU extended
    "weight_decay"  : [0.0, 1e-4, 1e-3],    # 0=none; 1e-4=clinical norm; 1e-3=strong L2
    "dropout"       : [0.2, 0.3, 0.5],      # [D4] 0.2–0.5 in classifier heads
    "scheduler_T0"  : [5, 10],              # Cosine restart period; T0=5 common in CXR
}
print("HP Search Space:")
for k,v in HP_SPACE.items(): print(f"  {k:<18}: {v}")
total = 1
for v in HP_SPACE.values(): total *= len(v)
print(f"Total combinations: {total}  |  Random sample: {MAX_TRIALS}")

HP Search Space:
  learning_rate     : [5e-05, 0.0001, 0.0005]
  batch_size        : [16, 32, 48]
  weight_decay      : [0.0, 0.0001, 0.001]
  dropout           : [0.2, 0.3, 0.5]
  scheduler_T0      : [5, 10]
Total combinations: 162  |  Random sample: 36


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — CheXpert Dataset + CLAHE Transforms [FIXED]
# ═══════════════════════════════════════════════════════════════════════
import cv2
from PIL import Image
import numpy as np

def apply_clahe(img_path):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        img = np.array(Image.open(str(img_path)).convert("L"))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return Image.fromarray(cv2.cvtColor(clahe.apply(img), cv2.COLOR_GRAY2RGB))

def csv_to_abs_path(raw_csv_path, data_root):
    """Strip the CheXpert-v1.0-small/ prefix — images live directly under data_root."""
    rel = raw_csv_path.replace("CheXpert-v1.0-small/", "")
    return Path(data_root) / rel

def filter_existing_files(df, data_root):
    """
    Fast filter — scans disk ONCE to build a set of known patient folders,
    then filters the dataframe in memory. Much faster than per-row .exists().
    """
    print("  Building valid patient index from disk (fast scan)...")
    train_dir = Path(data_root) / "train"
    valid_dir = Path(data_root) / "valid"

    # Collect all patient folder names that actually exist on disk
    existing_patients = set()
    for d in [train_dir, valid_dir]:
        if d.exists():
            for p in d.iterdir():
                if p.is_dir():
                    existing_patients.add(p.name)

    # Filter rows whose patient folder exists
    def patient_exists(raw_path):
        parts = raw_path.replace("CheXpert-v1.0-small/", "").split("/")
        return parts[1] if len(parts) > 1 else None  # parts[1] = patientXXXXX

    mask = df["Path"].apply(
        lambda p: patient_exists(p) in existing_patients
    )
    n_missing = (~mask).sum()
    print(f"  ✓ {mask.sum():,} valid  |  ✗ {n_missing:,} missing (dropped)")
    return df[mask].reset_index(drop=True)


class CheXpertDataset(Dataset):
    def __init__(self, df, image_root, transform=None, uncertain_policy="ones", use_clahe=True):
        self.df         = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform  = transform
        self.use_clahe  = use_clahe
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].fillna(0)
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(
            -1, 1 if uncertain_policy == "ones" else 0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = csv_to_abs_path(row["Path"], self.image_root)
        img      = apply_clahe(img_path) if self.use_clahe else Image.open(img_path).convert("RGB")
        lbl      = torch.tensor(row[LABEL_NAMES].values.astype(np.float32))
        if self.transform: img = self.transform(img)
        return img, lbl

def get_transforms(train=True):
    if train:
        return T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)), T.RandomHorizontalFlip(0.5),
                          T.RandomRotation(10), T.RandomAffine(degrees=0,translate=(0.05,0.05)),
                          T.ColorJitter(brightness=0.1,contrast=0.1), T.ToTensor(),
                          T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    return T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# ── Load, filter, split ────────────────────────────────────────────────
print("Loading CheXpert data...")
full_df = pd.read_csv(TRAIN_CSV)
print(f"  Raw CSV rows: {len(full_df):,}")

full_df = filter_existing_files(full_df, DATA_ROOT)   # ← drops missing files ONCE

hp_df, _         = train_test_split(full_df, test_size=1-HP_SEARCH_SUBSET, random_state=SEED)
hp_train, hp_val = train_test_split(hp_df,   test_size=0.15,               random_state=SEED)
print(f"  HP train: {len(hp_train):,}  |  HP val: {len(hp_val):,}  |  Full (clean): {len(full_df):,}")

Loading CheXpert data...


  Raw CSV rows: 223,414
  Building valid patient index from disk (fast scan)...


  ✓ 223,414 valid  |  ✗ 0 missing (dropped)
  HP train: 18,989  |  HP val: 3,352  |  Full (clean): 223,414


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — Model + Optimizer Builder
# ═══════════════════════════════════════════════════════════════════════
def build_model(hp):
    backbone = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    in_feat  = backbone.classifier.in_features
    backbone.classifier = nn.Identity()
    drop = float(hp["dropout"])
    head = nn.Sequential(nn.Dropout(drop), nn.Linear(in_feat, NUM_CLASSES))
    return nn.Sequential(backbone, head)

def build_opt_sched(model, hp, total_epochs=HP_SEARCH_EPOCHS):
    T0  = int(hp["scheduler_T0"])
    opt = Adam(model.parameters(), lr=float(hp["learning_rate"]), weight_decay=float(hp["weight_decay"]))
    sch = CosineAnnealingWarmRestarts(opt, T_0=T0, T_mult=1, eta_min=1e-6)
    return opt, sch

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Class Weight Computation
# ═══════════════════════════════════════════════════════════════════════
def compute_pos_weights(loader):
    pos, total = torch.zeros(NUM_CLASSES), 0
    for _, lbl in loader:
        pos += lbl.sum(0); total += lbl.size(0)
    return (total - pos) / (pos + 1e-5)

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Train / Validate Functions
# ═══════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train(); total_loss = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate_epoch(model, loader, criterion):
    model.eval(); total_loss, preds, labels = 0, [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(imgs); loss = criterion(logits, lbls)
            total_loss += loss.item()
            preds.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(lbls.cpu().numpy())
    p = np.concatenate(preds); l = np.concatenate(labels)
    aurocs = [roc_auc_score(l[:,i],p[:,i]) for i in range(NUM_CLASSES) if l[:,i].sum()>0]
    return total_loss/len(loader), np.mean(aurocs), p, l

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — HP Search Loop
# ═══════════════════════════════════════════════════════════════════════
keys   = list(HP_SPACE.keys())
combos = [dict(zip(keys,v)) for v in itertools.product(*HP_SPACE.values())]
random.shuffle(combos); combos = combos[:MAX_TRIALS]
print(f"\n{'='*68}\n  PHASE 1 — HP SEARCH  |  {MODEL_NAME.upper()}  |  {len(combos)} trials\n{'='*68}")

all_results = []
for t_idx, hp in enumerate(combos, 1):
    print(f"\n  Trial {t_idx:>2}/{len(combos)} | lr={hp['learning_rate']} | bs={hp['batch_size']} | wd={hp['weight_decay']} | drop={hp['dropout']} | T0={hp['scheduler_T0']}")
    bs      = int(hp["batch_size"])
    t_ds    = CheXpertDataset(hp_train, DATA_ROOT, get_transforms(True),  use_clahe=USE_CLAHE)
    v_ds    = CheXpertDataset(hp_val,   DATA_ROOT, get_transforms(False), use_clahe=USE_CLAHE)
    t_ld    = DataLoader(t_ds, bs, shuffle=True,  num_workers=4, pin_memory=True)
    v_ld    = DataLoader(v_ds, bs, shuffle=False, num_workers=4, pin_memory=True)
    model   = build_model(hp).to(DEVICE)
    pos_w   = compute_pos_weights(t_ld).to(DEVICE)
    crit    = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt,sch = build_opt_sched(model, hp)
    scaler  = torch.amp.GradScaler("cuda")
    epoch_logs, best_auroc, patience = [], 0.0, 0
    t_start = time.time()
    for ep in range(1, HP_SEARCH_EPOCHS+1):
        tr_loss             = train_one_epoch(model, t_ld, crit, opt, scaler)
        vl_loss, auroc, _,_ = validate_epoch(model, v_ld, crit)
        cur_lr = opt.param_groups[0]["lr"]
        if sch: sch.step()
        epoch_logs.append({"trial":t_idx,"epoch":ep,"train_loss":round(tr_loss,5),
                           "val_loss":round(vl_loss,5),"val_auroc":round(auroc,5),"lr":round(cur_lr,8)})
        print(f"    Ep{ep} | TrLoss:{tr_loss:.4f} VlLoss:{vl_loss:.4f} AUROC:{auroc:.4f} LR:{cur_lr:.2e}")
        if auroc > best_auroc: best_auroc=auroc; patience=0
        else:
            patience+=1
            if patience>=HP_PATIENCE: print(f"    ⚠ Early stop"); break
    elapsed = round(time.time()-t_start,1)
    result  = {"trial":t_idx,"model":MODEL_NAME,"best_auroc":round(best_auroc,5),
               "time_s":elapsed,"epoch_log":epoch_logs,**{f"hp_{k}":v for k,v in hp.items()}}
    all_results.append(result)
    with open(OUTPUT_DIR/f"hp_log_{MODEL_NAME}.json","w") as f: json.dump(all_results,f,indent=2)
    print(f"  ✓ Best AUROC: {best_auroc:.4f}  |  {elapsed}s")


  PHASE 1 — HP SEARCH  |  DENSENET121  |  36 trials

  Trial  1/36 | lr=0.0001 | bs=16 | wd=0.0001 | drop=0.2 | T0=10


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


  0%|          | 0.00/30.8M [00:00<?, ?B/s]

 43%|████▎     | 13.2M/30.8M [00:00<00:00, 138MB/s]

100%|██████████| 30.8M/30.8M [00:00<00:00, 168MB/s]

...

    Ep1 | TrLoss:1.0144 VlLoss:0.9741 AUROC:0.7185 LR:1.00e-04


    Ep2 | TrLoss:0.9616 VlLoss:0.9590 AUROC:0.7335 LR:9.05e-05


    Ep3 | TrLoss:0.9271 VlLoss:0.9521 AUROC:0.7462 LR:6.58e-05


    Ep4 | TrLoss:0.8922 VlLoss:0.9228 AUROC:0.7553 LR:3.52e-05


    Ep5 | TrLoss:0.8518 VlLoss:0.9221 AUROC:0.7588 LR:1.05e-05
  ✓ Best AUROC: 0.7588  |  825.2s


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Results Analysis + Optimal Config
# ═══════════════════════════════════════════════════════════════════════
rows = [{"trial":r["trial"],"best_auroc":r["best_auroc"],"time_s":r["time_s"],
         **{k.replace("hp_",""):v for k,v in r.items() if k.startswith("hp_")}}
        for r in all_results]
summary = pd.DataFrame(rows).sort_values("best_auroc", ascending=False)
summary.to_csv(OUTPUT_DIR/f"hp_summary_{MODEL_NAME}.csv", index=False)
print(f"\nTOP-10 TRIALS:\n{summary.head(10).to_string(index=False)}")

hp_cols = [c for c in summary.columns if c not in ["trial","best_auroc","time_s"]]
sens_lines = ["SENSITIVITY ANALYSIS"]
for col in hp_cols:
    grp = summary.groupby(col)["best_auroc"].mean().sort_values(ascending=False)
    sens_lines.append(f"  {col}:")
    for val,auroc in grp.items():
        bar = "█"*max(1,int((auroc-grp.min())/(grp.max()-grp.min()+1e-9)*20))
        sens_lines.append(f"    {str(val):>10}  {bar:<22} mean AUROC={auroc:.4f}")
sens_text = "\n".join(sens_lines)
print(f"\n{sens_text}")
with open(OUTPUT_DIR/f"sensitivity_{MODEL_NAME}.txt","w") as f: f.write(sens_text)

best_row = summary.iloc[0]
optimal  = {col:best_row[col] for col in hp_cols}
optimal["val_auroc_phase1"] = best_row["best_auroc"]
with open(OUTPUT_DIR/f"optimal_{MODEL_NAME}.json","w") as f: json.dump(optimal,f,indent=2)
print(f"\nOPTIMAL CONFIG: {optimal}")


TOP-10 TRIALS:
 trial  best_auroc  time_s  learning_rate  batch_size  weight_decay  dropout  scheduler_T0
    10     0.75942   855.6        0.00010          16        0.0010      0.2             5
    36     0.75876   825.2        0.00010          16        0.0000      0.2             5
     5     0.75629   648.5        0.00050          48        0.0000      0.2             5
     1     0.75438   874.7        0.00010          16        0.0001      0.2            10
    29     0.75268   856.6        0.00010          16        0.0010      0.2            10
    15     0.75183   633.0        0.00010          48        0.0001      0.2             5
    22     0.75080   650.2        0.00010          32        0.0000      0.2             5
     3     0.75060   655.1        0.00010          48        0.0010      0.3             5
    27     0.75018   662.3        0.00010          32        0.0000      0.5             5
    17     0.74904   847.5        0.00005          16        0.0000      0

In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Final Training
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'='*68}\n  PHASE 2 — FINAL TRAINING  |  {MODEL_NAME.upper()}  |  {FINAL_EPOCHS} epochs\n{'='*68}")
opt_hp = {k:v for k,v in optimal.items() if k!="val_auroc_phase1"}

full_train, full_val = train_test_split(full_df, test_size=0.05, random_state=SEED)
bs_f  = int(opt_hp["batch_size"])
ft_ds = CheXpertDataset(full_train, DATA_ROOT, get_transforms(True),  use_clahe=USE_CLAHE)
fv_ds = CheXpertDataset(full_val,   DATA_ROOT, get_transforms(False), use_clahe=USE_CLAHE)
ft_ld = DataLoader(ft_ds, bs_f, shuffle=True,  num_workers=4, pin_memory=True)
fv_ld = DataLoader(fv_ds, bs_f, shuffle=False, num_workers=4, pin_memory=True)

final_model = build_model(opt_hp).to(DEVICE)
final_pos_w = compute_pos_weights(ft_ld).to(DEVICE)
final_crit  = nn.BCEWithLogitsLoss(pos_weight=final_pos_w)
final_opt, final_sch = build_opt_sched(final_model, opt_hp, FINAL_EPOCHS)
final_scaler = torch.amp.GradScaler("cuda")
ckpt_path    = OUTPUT_DIR / f"best_{MODEL_NAME}.pth"

final_log, best_auroc, best_epoch, patience = [], 0.0, 0, 0
for ep in range(1, FINAL_EPOCHS+1):
    tr_loss              = train_one_epoch(final_model, ft_ld, final_crit, final_opt, final_scaler)
    vl_loss, auroc, _,_  = validate_epoch(final_model, fv_ld, final_crit)
    cur_lr = final_opt.param_groups[0]["lr"]
    if final_sch: final_sch.step()
    final_log.append({"epoch":ep,"train_loss":round(tr_loss,5),"val_loss":round(vl_loss,5),
                      "val_auroc":round(auroc,5),"lr":round(cur_lr,8)})
    marker = ""
    if auroc > best_auroc:
        best_auroc, best_epoch, patience = auroc, ep, 0
        torch.save({"epoch":ep,"model_state_dict":final_model.state_dict(),
                    "auroc":auroc,"optimal_hp":opt_hp,"history":final_log}, ckpt_path)
        marker = " ✓ NEW BEST"
    else:
        patience+=1; marker=f" ({patience}/{FINAL_PATIENCE})"
        if patience>=FINAL_PATIENCE: print(f"  ⚠ Early stop epoch {ep}"); break
    print(f"  Ep{ep:>2}/{FINAL_EPOCHS} | TrLoss:{tr_loss:.4f} VlLoss:{vl_loss:.4f} AUROC:{auroc:.4f} LR:{cur_lr:.2e}{marker}")
pd.DataFrame(final_log).to_csv(OUTPUT_DIR/f"final_log_{MODEL_NAME}.csv", index=False)
print(f"\nPhase 2 done | Best AUROC: {best_auroc:.4f} at epoch {best_epoch}")


  PHASE 2 — FINAL TRAINING  |  DENSENET121  |  15 epochs


  Ep 1/15 | TrLoss:0.9611 VlLoss:0.9279 AUROC:0.7489 LR:1.00e-04 ✓ NEW BEST


  Ep 2/15 | TrLoss:0.9351 VlLoss:0.9306 AUROC:0.7532 LR:9.05e-05 ✓ NEW BEST


  Ep 3/15 | TrLoss:0.9189 VlLoss:0.9035 AUROC:0.7678 LR:6.58e-05 ✓ NEW BEST


  Ep 4/15 | TrLoss:0.9016 VlLoss:0.8830 AUROC:0.7774 LR:3.52e-05 ✓ NEW BEST


  Ep 5/15 | TrLoss:0.8842 VlLoss:0.8682 AUROC:0.7852 LR:1.05e-05 ✓ NEW BEST


  Ep 6/15 | TrLoss:0.9217 VlLoss:0.9041 AUROC:0.7649 LR:1.00e-04 (1/5)


  Ep 7/15 | TrLoss:0.9171 VlLoss:0.9021 AUROC:0.7650 LR:9.05e-05 (2/5)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — Per-Label Evaluation
# ═══════════════════════════════════════════════════════════════════════
ckpt = torch.load(ckpt_path)
final_model.load_state_dict(ckpt["model_state_dict"])
_, _, fp, fl = validate_epoch(final_model, fv_ld, final_crit)
print(f"\n{'='*68}\n  FINAL PER-LABEL RESULTS — {MODEL_NAME.upper()}\n{'='*68}")
per_label = []
for i, name in enumerate(LABEL_NAMES):
    if fl[:,i].sum()==0: continue
    auroc = roc_auc_score(fl[:,i], fp[:,i])
    ap    = average_precision_score(fl[:,i], fp[:,i])
    f1    = f1_score(fl[:,i], (fp[:,i]>=0.5).astype(int), zero_division=0)
    per_label.append({"label":name,"auroc":round(auroc,4),"ap":round(ap,4),"f1":round(f1,4)})
    print(f"  {name:<30}  {auroc:>7.4f}  {ap:>7.4f}  {f1:>7.4f}")
ma = np.mean([r["auroc"] for r in per_label])
print(f"\n  MEAN AUROC: {ma:.4f}")
pd.DataFrame(per_label).to_csv(OUTPUT_DIR/f"per_label_{MODEL_NAME}.csv", index=False)
print(f"\n✓ All outputs saved to /kaggle/working/")